Steps to preprocess MODIS data for phenology tracking. Note: GEE runs into memory issues when you try manipulating any of the big data for plotting/viewing/etc. So I am exporting these as geotiffs and reading them in as rasters to python to be able to look at the processing steps.

Author: Tayler LaSharr 

Date: September 2024 (LW changes April 23 2026)

In [14]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code


import ee
import geemap
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import os
import pprint
import time
import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )

## initialize the Earth Engine module
#ee.Authenticate()  # run once; follow the browser flow
# during authentication, the profile is "ee-lwilde2", but the project is westwidemuledeer - can throw errors if you try to use the wrong project name, so make sure to use the correct one
ee.Initialize(project="westwidemuledeer")

In [15]:
## some quick settings 

## must run 2003 to 2012 seperately from 2013 to 2025 because of the change in MODIS products (MOD13Q1 to MYD13Q1) and the way I am downloading the data (using the product name as a filter for which years to download)
## generate Yearly Date Ranges
start_date_export = '2013-01-01'
end_date_export = '2025-12-31'

## get the time period we care about
start_date_data = '2013-01-01'
end_date_data = '2025-12-31'

## NOT RUN if years are before 2013, use MOD13Q1, if after 2012, use MYD13Q1
## SOURCE = 'MODIS/061/MOD13Q1' ## for 2003-2012 
## SOURCE = 'MODIS/061/MOD09Q1' ## for 2013-2025


## then also, define the area of interest, we will go state by state

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## some snow in winter

## get a bounding box for clipping purposes
# bounding_box = md_cooridor.geometry().bounds()

## alternatively, you can create a bounding box from coordinates.
# bounding_box = ee.Geometry.Rectangle([-112.125449, 40.010084, -102.721152, 45.717045])
# bounding_box = ee.Geometry.Rectangle([-111.232888, 40.859746, -103.876160, 45.140904]) ## wyoming
# bounding_box = ee.Geometry.Rectangle([-114.522, 36.5979, -109.416, 42.5016]) ## utah
# bounding_box = ee.Geometry.Rectangle([-111.626918, 43.275844, -109.512941, 45.310065])

### CHOOSE STATE FOULDER NAME ###
STATE = 'FULL_WESTERN_US' ## full western US

### CHOOSE STATE BOUNDING BOX ###
bounding_box = ee.Geometry.Rectangle([-125.03826637268067, 31.024805248039495, -101.48357887268067, 50.04311301899164])  ## Full Western US
# bounding_box = ee.Geometry.Rectangle([-111.0546, 40.9950, -104.0522, 45.0059])  ## wyoming
# bounding_box = ee.Geometry.Rectangle([-116.0490, 44.3582, -104.0400, 49.0011])  ## montana
# bounding_box = ee.Geometry.Rectangle([-117.2413, 41.9880, -111.0440, 49.0011])  ## idaho
# bounding_box = ee.Geometry.Rectangle([-124.7631, 45.5435, -116.9150, 49.0024])  ## washington
# bounding_box = ee.Geometry.Rectangle([-124.5662, 41.9918, -116.4633, 46.2615])  ## oregon
# bounding_box = ee.Geometry.Rectangle([-124.4096, 32.5343, -114.1312, 42.0095])  ## california
# bounding_box = ee.Geometry.Rectangle([-120.0057, 35.0018, -114.0396, 42.0022])  ## nevada
# bounding_box = ee.Geometry.Rectangle([-114.0529, 36.9979, -109.0452, 42.0017])  ## utah
# bounding_box = ee.Geometry.Rectangle([-114.8165, 31.3322, -109.0452, 37.0042])  ## arizona
# bounding_box = ee.Geometry.Rectangle([-109.0502, 31.3322, -103.0020, 37.0001])  ## new mexico
# bounding_box = ee.Geometry.Rectangle([-109.0602, 36.9924, -102.0424, 41.0034])  ## colorado




In [16]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '11-01'
end_month_day = '03-30'

## temporal window to deal with snow
required_snow_free_days=3
snow_free_window_size=5




Obtain Data.

This step reads in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons) and define the time intervals that you want to classify winter as anbd the time series for the data you are interested in. This differs very slighly from the R code and uses the day-months instead of the Julian date. This difference is solely to help deal with leap years in systems where winter may end before the end of Feburary when I am working in desert systems.

In [17]:



## load the MODIS NDVI data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band to the modis collection
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data to reproject the bounding box
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )

modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))
print(target_projection)


{'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}


Get an image collection of the filtered NDVI data and an image collection of the snow cover data.

First function removes low-quality observations with the QA band. This function will create an image collection called 'NDVIFiltered' that has filtered NDVI data during the time period you are interested in (i.e., removes poor quality pixels, snow, and clouds). This is the equivalent of the 'NDVI' rasters produced in step 1b of the R code.

The second function creates a snow mask; this differs from the mask that Jerod and Shannon had made, this will instead create a mask if a NDVI pixel occurs at a point in time where 2 weeks prior and 1 week after the date of the NDVI image has average snow cover that exceeds a set threshold.


In [18]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow
def quality_and_state_mask(image):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  ## clear or cloudy, exclude mixed and not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  ## no cloud shadow
    is_land = state.rightShift(3).bitwiseAnd(7).eq(1)  ## land (not in Jerod's code)
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  ## no snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  ## highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(good_quality).And(no_snow_ice).And(is_land)

    ## apply the mask to the NDVI band specifically
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    ## then, get rid of outliers
    ndvi_masked =  ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(2)))

    ## return the masked image
    return ndvi_masked

## this function is to extract snow information from the State band of the MODIS data
def extract_snow(image):
    
    ## get the state band
    state = image.select('State')

    ## bit 12 represents snow/ice
    snow_mod35 = state.bitwiseAnd(1 << 12).neq(0)  # 1 if snow/ice is present, 0 otherwise

    ## bit 16 represents internal snow
    snow_internal = state.bitwiseAnd(1 << 16).neq(0)  # 1 if internal snow is present, 0 otherwise

    ## combine the MOD35 and internal snow
    snow_combined = snow_mod35.Or(snow_internal)  # 1 if either MOD35 or internal snow is present

    ## add the snow band to the image
    snow_band = snow_combined.rename('SnowCovered')

    ## carry over the system:time_start property from the original image (this is not saved in the State band, so you need to explicitly call it back in)
    ## and make sure it stays in the native projection

    snow_band = snow_band.set('system:time_start', image.get('system:time_start'))
    return snow_band


## apply the quality mask to the MODIS collection
NDVIFiltered = modis_ndvi.map(lambda img: quality_and_state_mask(img))

## apply the snow extraction function to the MODIS collection
SnowCover = modis_ndvi.map(extract_snow)

## just check on a couple of things
# print('Filtered MODIS information:', NDVIFiltered.getInfo())
print("Bands available in the first image:", NDVIFiltered.first().bandNames().getInfo())
print("Number of Images:", NDVIFiltered.size().getInfo())

# print('Filtered MODIS Snow information (NDVI):', SnowCover.getInfo())
print("Bands available in the first image (Snow):", SnowCover.first().bandNames().getInfo())
print("Number of Images:", SnowCover.size().getInfo())



Bands available in the first image: ['NDVI']
Number of Images: 598
Bands available in the first image (Snow): ['SnowCovered']
Number of Images: 598


Steps 1 - 4. Calculations. Reclassify outliers, reclassify winter values, check if a pixel has been snow-free for a set number of consecutive images, mask pizels that are in winter or covered by snow, apply median filter.

Step 1 in this section are the same as the steps in Merkle and Albeke's code for processing NDVI. Step 2 deviates from their methods. Merkle and Albeke methods require a pixel = to be snow free for 3 consecutive time steps in order to retain the NDVI value. This process doesn't work great in the more arid environments, so I shifted that methodology here. Instead, I required a minimum number of snow free days within a moving window. For now, I used a 5-image moving window with a requirement of at least 3 snow-free images within that window. This allows for brief snowfall events to be overlooked while still ensuring sufficient snow-free period in more temperate systems.

Step 3 is the same as the Merkle and Albeke methods.

In [19]:
## step 1: reclassify outliers
def ndviCALC_step1(NDVICollection):
    
    ## reclassify outliers (mask NDVI values less than 0 and greater than 2)
    def reclassify_outliers(image):
        return image.updateMask(image.select('NDVI').gte(0).And(image.select('NDVI').lte(2)))

    NDVICollection = NDVICollection.map(reclassify_outliers)
    return NDVICollection

## step 2: check if a pixel has been snow-free for at least N days within a moving window and
## update the snowcover collection
def ndviCALC_step2(image_collection, snow_free_window_size = 5, required_snow_free_days = 3):
    
    def calculate_snow_free_for_image(index):
        
        ## define the start and end of the N-image window
        start_index = ee.Number(index).subtract(snow_free_window_size - 1).max(0)
        end_index = ee.Number(index).add(1)

        ## get the moving window of images and select the 'Snow' band
        window_images = image_collection.toList(image_collection.size()).slice(start_index, end_index)
        window_collection = ee.ImageCollection.fromImages(window_images).select('SnowCovered')

        ## calculate the number of snow-free days (where Snow == 0) within the window
        snow_free_days = window_collection.map(lambda img: img.eq(0)).sum()

        ## get the current image
        current_image = ee.Image(image_collection.toList(image_collection.size()).get(index))

        ## create a binary mask where pixels are considered snow-covered if they do not meet the snow-free threshold
        snow_covered_mask = snow_free_days.lt(required_snow_free_days)  # Flip the threshold logic
        
        ## apply this binary mask to the current image
        result_image = ee.Algorithms.If(
            ee.Number(index).lt(snow_free_window_size - 1),
            current_image.select('SnowCovered'),  # use original image for early indices
            snow_covered_mask.rename('SnowCovered').set('system:time_start', current_image.get('system:time_start'))
        )

        ## this returns an image that has a binary 1/0 where 1 represents 'snow covered' pixels
        return result_image

    ## apply this function across each image in the collection
    indices = ee.List.sequence(0, image_collection.size().subtract(1))
    processed_collection = indices.map(lambda i: calculate_snow_free_for_image(i))
    
    return ee.ImageCollection.fromImages(processed_collection)
    

## run first 5 steps sequentially
NDVIProcessed1 = ndviCALC_step1(NDVIFiltered)
print("Step 1 complete: Removed outliers from data.")
print("Number of Images:", NDVIProcessed1.size().getInfo())

SnowCover_MovingWindow = ndviCALC_step2(SnowCover, required_snow_free_days=3, snow_free_window_size=5)
print("Step 2 complete: Apply a mask based on the snow cover image collection.")
print("Number of Images:", SnowCover_MovingWindow.size().getInfo())


Step 1 complete: Removed outliers from data.
Number of Images: 598
Step 2 complete: Apply a mask based on the snow cover image collection.
Number of Images: 598


Steps 5 - 8. Calculate the baseline value, mask NDVI values that are covered with snow, and floor values below the baseline and covered by snow during the winter period.

These steps also differ slightly from Jerod's methods. Step 4a and 4b generally acheives the same thing as Jerod and Shannon's methods, but I shifted steps 5 - 7. Here, it adds a band of yes/no winter and yes/no snow (based on both the snow mask image collectoin and the moving window snow mask image collection). Finally, it applied those masks and replaced any masked snow values with the baseline image value. You can decide if you want to floor all winter values as well.

In [20]:
## step 3: add a winter mask
def ndviCALC_step3(NDVICollection, start_month_day='11-01', end_month_day='03-30'):

    ## parse the start and end month-day strings
    start_month, start_day = map(int, start_month_day.split('-'))
    end_month, end_day = map(int, end_month_day.split('-'))
    
    def add_winter_mask(image):
        
        ## get the date and year of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')

        ## get the reference pro0jection
        reference_projection = NDVICollection.first().projection()
                
        ## define winter start and end dates for the current year
        winter_start = ee.Date.fromYMD(year, start_month, start_day)
        winter_end = ee.Date.fromYMD(year, end_month, end_day)
        
        ## create a condition to determine if the image is in the winter period
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(
            date.difference(winter_end, 'day').lte(0)
        )
        
        ## keep it in the correct projection
        winter_mask = ee.Image(is_in_winter).rename('winter_mask').reproject(
            crs=reference_projection.crs(),
            scale=reference_projection.nominalScale()
        )

        ## add the winter mask as a new band
        return image.addBands(winter_mask)
    
    ## apply the winter mask to each image in the collection
    NDVICollection_with_winter_mask = NDVICollection.map(add_winter_mask)
    
    return NDVICollection_with_winter_mask

## step 4: add a snow mask layer
def ndviCALC_step4(NDVICollection, SnowCover_MovingWindow, SnowCover):

    def add_snow_masks(ndvi_image):
        
        ## get the NDVI date
        ndvi_date = ndvi_image.get('system:time_start')

        ## find the corresponding snow mask by matching dates
        moving_window_mask = SnowCover_MovingWindow.filter(ee.Filter.eq('system:time_start', ndvi_date)).first()
        original_snow_mask = SnowCover.filter(ee.Filter.eq('system:time_start', ndvi_date)).first()

        ## if there's no matching snow mask, use a default mask with no snow (0)
        moving_window_mask = ee.Image(ee.Algorithms.If(
            moving_window_mask, moving_window_mask.select('SnowCovered'), ee.Image.constant(0).rename('SnowCovered')
        ))
        
        original_snow_mask = ee.Image(ee.Algorithms.If(
            original_snow_mask, original_snow_mask.select('SnowCovered'), ee.Image.constant(0).rename('SnowCovered')
        ))
        
        combined_snow_mask = moving_window_mask.Or(original_snow_mask).rename('SnowCovered')

        ## apply the commbined band to the NDVI band
        ndvi_image_with_combined_mask = ndvi_image.addBands(combined_snow_mask)

        ## add the snow mask as a new band to the NDVI image
        return ndvi_image_with_combined_mask
    
    ## apply the snow mask addition to each image in the NDVI collection
    NDVICollection_with_masks = NDVICollection.map(add_snow_masks)
    
    return NDVICollection_with_masks

## step 5: calculate the floor (0.025 quantile) without masking cells in the baseline image that have masked cells elsewhere in the image collection
## I was running into a lot of issues here trying to used the already masked image collections (NDVIProcessed4)
## so I am rerunning the masks. This works, I have no idea why using the existing image collections did not.
## first mask things

def ndviCALC_step5a(image):

    ## apply quality and snow masks
    quality_masked = quality_and_state_mask(image)
    snow_mask1 = SnowCover_MovingWindow.filter(ee.Filter.eq('system:time_start', image.get('system:time_start'))) \
        .first().select('SnowCovered').eq(0)  # Snow-free pixels in MovingWindow
    snow_mask2 = SnowCover.filter(ee.Filter.eq('system:time_start', image.get('system:time_start'))) \
        .first().select('SnowCovered').eq(0)  # Snow-free pixels in SnowCover

    combined_snow_mask = snow_mask1.And(snow_mask2)  # Snow-free in both masks
    
    ## get rid of negative values and reclassify outliers
    masked_image = quality_masked.updateMask(combined_snow_mask)  # Apply combined snow mask
    valid_ndvi = masked_image.updateMask(masked_image.select('NDVI').gte(0).And(masked_image.select('NDVI').lte(2)))

    return valid_ndvi

def ndviCALC_step5(masked_collection, quantile_value=2.5):

    quantile_fraction = quantile_value / 100.0

    ## get the projection to retain
    reference_projection = masked_collection.first().projection()

    ## calculate the 0.025 quantile across the entire time series for each pixel
    baseline_image = masked_collection.select('NDVI').reduce(
        ee.Reducer.percentile([quantile_fraction * 100])
    ).rename('NDVI_baseline')

    ## keep in the original projection
    baseline_image = baseline_image.reproject(
        crs=reference_projection.crs(),
        scale=reference_projection.nominalScale()
    )

    return baseline_image

## step 5: apply corrections to the NDVI collection based on snow cover and winter periods
## you can turn on and off if you want it to replace all the winter values as the baseline
def ndviCALC_step6(NDVICollection, baseline_ndvi, restrict_to_winter=False):

    def apply_corrections(image):

        ## select masks and NDVI band
        winter_mask = image.select('winter_mask').reduce(ee.Reducer.max()).rename('winter_mask') ## ensure single band
        snow_mask = image.select('SnowCovered').reduce(ee.Reducer.max()).rename('SnowCovered') ## ensure single band
        ndvi = image.select('NDVI') ## NDVI values
        
        ## decide whether to restrict corrections to winter months
        if restrict_to_winter:
            correction_condition = winter_mask.And(snow_mask)
        else:
            correction_condition = snow_mask
        
        ## apply corrections, set NDVI to baseline where conditions are met
        corrected_ndvi = ndvi.where(correction_condition.Or(ndvi.lt(baseline_ndvi.select('NDVI_baseline'))), baseline_ndvi.select('NDVI_baseline'))
        corrected_ndvi = corrected_ndvi.unmask(baseline_ndvi.select('NDVI_baseline'))
        
        ## add the corrected NDVI as an updated band
        return image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)
    
    ## apply corrections to each image in the NDVI collection
    corrected_NDVICollection = NDVICollection.map(apply_corrections)
    
    return corrected_NDVICollection


# Step 3: Add the winter mask to each image
NDVIProcessed3 = ndviCALC_step3(NDVIProcessed1, start_month_day, end_month_day)
print("Step 3 complete: Add winter mask.")
print("Number of Images:", NDVIProcessed3.size().getInfo())

# Step 4: Add the snow mask to each image
NDVIProcessed4 = ndviCALC_step4(NDVIProcessed3, SnowCover_MovingWindow, SnowCover)
print("Step 4 complete: Add snow mask.")
print("Number of Images:", NDVIProcessed4.size().getInfo())

# Step 5: Get the baseline image
step5a_tmp_imagecollection = modis_ndvi.map(ndviCALC_step5a) ## run the masks
baseline_ndvi_image = ndviCALC_step5(step5a_tmp_imagecollection, quantile_value=2.5)
print("Step 5 commplete: Baseline NDVI Image (0.025 quantile) calculated successfully.")
# print("Baseline NDVI Image Info:", baseline_ndvi_image.getInfo())

# Step 6: Apply corrections based on winter and snow mask
corrected_NDVI = ndviCALC_step6(NDVIProcessed4, baseline_ndvi_image, restrict_to_winter=False)
print("Step 6 complete: Add baseline values where snow exists.")
print("Number of Images:", corrected_NDVI.size().getInfo())
print("Bands available in final processed NDVI image collection:", corrected_NDVI.first().bandNames().getInfo())



Step 3 complete: Add winter mask.
Number of Images: 598
Step 4 complete: Add snow mask.
Number of Images: 598
Step 5 commplete: Baseline NDVI Image (0.025 quantile) calculated successfully.
Step 6 complete: Add baseline values where snow exists.
Number of Images: 598
Bands available in final processed NDVI image collection: ['NDVI', 'winter_mask', 'SnowCovered']


In [21]:
def print_projection_info(collection, step_name):
    
    ## get the projection of the first image in the collection
    projection = collection.first().projection()
    crs = projection.crs().getInfo()  # Get the CRS
    scale = projection.nominalScale().getInfo()  # Get the scale in meters
    
    print(f"Step {step_name} Projection:")
    print(f"  CRS: {crs}")
    print(f"  Scale: {scale} meters\n")

def print_image_projection(image, name="Single Image"):
    
    ## get the projection of the image
    projection = image.projection()
    
    ## retrieve CRS and scale
    crs = projection.crs().getInfo()
    scale = projection.nominalScale().getInfo()

    ## print the results
    print(f"{name} Projection:")
    print(f"  CRS: {crs}")
    print(f"  Scale: {scale} meters\n")


# print_projection_info(NDVIProcessed1, "1 (Outlier Removal)")
# print_projection_info(NDVIProcessed3.select('NDVI'), "3 (Add Winter Mask) - NDVI")
print_projection_info(NDVIProcessed4.select('winter_mask'), "4 (Add Winter Mask) - Winter")
print_projection_info(NDVIProcessed4.select('NDVI'), "NDVI")
print_projection_info(NDVIProcessed4.select('SnowCovered'), "Snow Cover")


# print_image_projection(baseline_ndvi_image, name="Baseline NDVI Image")

# print_projection_info(corrected_NDVI.select('NDVI'), "Final Corrected NDVI")

# print("Bands available in the final image (NDVI):", corrected_NDVI.first().bandNames().getInfo())
# print("Number of Images:", corrected_NDVI.size().getInfo())

Step 4 (Add Winter Mask) - Winter Projection:
  CRS: SR-ORG:6974
  Scale: 231.65635826395828 meters

Step NDVI Projection:
  CRS: SR-ORG:6974
  Scale: 231.65635826395828 meters

Step Snow Cover Projection:
  CRS: SR-ORG:6974
  Scale: 231.65635826395828 meters



In [22]:
print("Bands available in the final image (NDVI):", NDVIProcessed4.first().bandNames().getInfo())
print("Number of Images:", NDVIProcessed4.size().getInfo())
print("Info from first image:", NDVIProcessed4.first().getInfo())

Bands available in the final image (NDVI): ['NDVI', 'winter_mask', 'SnowCovered']
Number of Images: 598
Info from first image: {'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [20150, 9418], 'origin': [34967, 9291], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}, {'id': 'winter_mask', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 0}, 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395828, 0, 0, 0, -231.65635826395828, 0]}, {'id': 'SnowCovered', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 1}, 'dimensions': [20150, 9418], 'origin': [34967, 9291], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}], 'version': 1749179928117924, 'id': 'MODIS/061/MOD09Q1/2013_01_01', 'properties': {'system:footprint': {'geodesic': False, 'crs

This next step is also new. Because we aren't just flooring across the entire winter, this next step adds another general smoothing function that will correct the pixel valuye to the baseline if the both the neighboring images are at the baseline.

Check that the correct projection was retained through all steps.

In [23]:
# ## remove all the image collections we no longer need (this helps reduce memory in GEE)
# NDVIProcessed1 = None 
# NDVIProcessed3 = None
# NDVIProcessed4 = None
# step5a_tmp_imagecollection = None  


Code to export the data. We export the data in grids because of memory issues with GEE. We will have to merge and mosaic them back together when we read in the geotiffs to scale and fit curves.

In [24]:
## create a grid from bounding box
def create_grid(bounding_box, rows, cols):
    bounds = bounding_box.bounds().getInfo()['coordinates'][0]
    min_lon, min_lat = bounds[0]
    max_lon, max_lat = bounds[2]

    lon_step = (max_lon - min_lon) / cols
    lat_step = (max_lat - min_lat) / rows

    grid = []
    for i in range(rows):
        for j in range(cols):
            lon1 = min_lon + lon_step * j
            lat1 = min_lat + lat_step * i
            lon2 = lon1 + lon_step
            lat2 = lat1 + lat_step
            tile = ee.Geometry.Rectangle([lon1, lat1, lon2, lat2])
            grid.append(tile)

    return grid

## grid information
rows, cols = 3, 3
grid = create_grid(bounding_box, rows, cols)

def print_grid_info(grid):
    print("Grid Information:")
    for idx, tile in enumerate(grid):
        bounds = tile.bounds().coordinates().getInfo()
        crs = tile.projection().crs().getInfo()
        print(f"Tile {idx + 1}:")
        print(f"  Bounds: {bounds}")
        print(f"  CRS: {crs}")
        print()


In [25]:

## generate yearly date ranges
def generate_yearly_date_ranges(start_date, end_date):
    start_year = datetime.strptime(start_date, "%Y-%m-%d").year
    end_year = datetime.strptime(end_date, "%Y-%m-%d").year
    yearly_ranges = []

    for year in range(start_year, end_year + 1):
        start_of_year = f"{year}-01-01"
        end_of_year = f"{year}-12-31"
        yearly_ranges.append((start_of_year, end_of_year))

    return yearly_ranges



yearly_ranges = generate_yearly_date_ranges(start_date_export, end_date_export)
print("Yearly Ranges:", yearly_ranges)

## ensure bands are cast to float
def cast_bands_to_float(image):
    ndvi_band = image.select('NDVI').toFloat().reproject(crs='EPSG:4326', scale=231.65635826395828)
    # winter_mask_band = image.select('winter_mask').toFloat().reproject(crs='EPSG:4326', scale=231.65635826395828)
    # snow_covered_band = image.select('SnowCovered').toFloat().reproject(crs='EPSG:4326', scale=231.65635826395828)
    # return ee.Image.cat([ndvi_band, winter_mask_band, snow_covered_band]).copyProperties(image, image.propertyNames())
    return ee.Image.cat([ndvi_band]).copyProperties(image, image.propertyNames())

## export function with WGS84 projection
def export_yearly_tiles_with_bands(image_collection, bands, grid, yearly_ranges, file_name='NDVI_', folder_name='NDVI_Exports'):
    for start_date, end_date in yearly_ranges:
        yearly_collection = image_collection.filterDate(start_date, end_date).select(bands).map(cast_bands_to_float)
        target_scale = 231.65635826395828
        
        print(f"Exporting data for year: {start_date} to {end_date}")
        
        ndvi_bands = yearly_collection.toBands()
        
        for tile_index, tile in enumerate(grid):
            export_image = ndvi_bands.clip(tile)
            
            export_task = ee.batch.Export.image.toDrive(
                image=export_image,
                description=f'{file_name}_{start_date[:4]}_Tile_{tile_index + 1}',
                scale=target_scale,
                region=tile,
                crs='EPSG:4326',  # Use WGS84 projection
                fileFormat='GeoTIFF',
                folder=folder_name
            )
            export_task.start()
            print(f"Started export task for Year {start_date[:4]} Tile {tile_index + 1}")
            time.sleep(5)  ## add 5 second pause between task starts


## apply the export function
export_yearly_tiles_with_bands(corrected_NDVI, bands=['NDVI'], grid=grid, yearly_ranges=yearly_ranges, file_name=STATE, folder_name='NDVI_Exports')


Yearly Ranges: [('2013-01-01', '2013-12-31'), ('2014-01-01', '2014-12-31'), ('2015-01-01', '2015-12-31'), ('2016-01-01', '2016-12-31'), ('2017-01-01', '2017-12-31'), ('2018-01-01', '2018-12-31'), ('2019-01-01', '2019-12-31'), ('2020-01-01', '2020-12-31'), ('2021-01-01', '2021-12-31'), ('2022-01-01', '2022-12-31'), ('2023-01-01', '2023-12-31'), ('2024-01-01', '2024-12-31'), ('2025-01-01', '2025-12-31')]
Exporting data for year: 2013-01-01 to 2013-12-31
Started export task for Year 2013 Tile 1
Started export task for Year 2013 Tile 2
Started export task for Year 2013 Tile 3
Started export task for Year 2013 Tile 4
Started export task for Year 2013 Tile 5
Started export task for Year 2013 Tile 6
Started export task for Year 2013 Tile 7
Started export task for Year 2013 Tile 8
Started export task for Year 2013 Tile 9
Exporting data for year: 2014-01-01 to 2014-12-31
Started export task for Year 2014 Tile 1
Started export task for Year 2014 Tile 2
Started export task for Year 2014 Tile 3
St

In [26]:
# check the status of the export tasks at: https://code.earthengine.google.com/tasks